# WIT-1024: выравнивание BLOOMZ и DINOv2

Воспроизведение эксперимента Huh et al. (2024) на Kaggle. Перед запуском включите *Settings → Accelerator → GPU T4 x2* и *Settings → Internet → On*, затем *Save Version → Save & Run All*. Расчёт занимает около часа, результаты появятся в `/kaggle/working/wit1024` и в архиве `wit1024_results.zip`.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/velton00/rep-convergence-recsys.git"

# архив проекта, добавленный как Kaggle Dataset, используется вместо клонирования
uploaded = next(Path("/kaggle/input").rglob("pyproject.toml"), None)
if uploaded is not None:
    !cp -r "{uploaded.parent}" repo
else:
    !git clone -q {REPO_URL} repo
%pip install -q ./repo

In [ ]:
!python repo/scripts/run_wit1024.py --out /kaggle/working/wit1024

In [ ]:
import pandas as pd
from IPython.display import Image, display

out = Path("/kaggle/working/wit1024")
table = pd.read_csv(out / "alignment.csv")
llms = table.sort_values("llm_params_b")["llm"].unique()
lvms = table.sort_values("lvm_params_m")["lvm"].unique()
display(table.pivot(index="llm", columns="lvm", values="score").loc[llms, lvms].round(3))
display(Image(out / "figures" / "alignment_vs_size.png", width=640))

In [ ]:
import zipfile

with zipfile.ZipFile("/kaggle/working/wit1024_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for path in out.rglob("*"):
        if path.is_file() and "features" not in path.parts:
            z.write(path, path.relative_to(out.parent))